In [ ]:
# Activity 1 Try out

activity_date = "2020-01-01"

print(activity_date)
print(type(activity_date))

2020-01-01
<class 'str'>


In [ ]:
# Activity 2 Try out

confirmed_cases = [600_000, 50, 10_000]

def classify_case_burden(confirmed_cases):
    if confirmed_cases >= 1_000_000:
        return "Very high"
    elif confirmed_cases >= 100_000:
        return "High"
    elif confirmed_cases >= 10_000:
        return "Moderate"
    else:
        return "Low"

for cases in confirmed_cases :
    print(f"{cases:,} cases:", classify_case_burden(cases))

600,000 cases: High
50 cases: Low
10,000 cases: Moderate


In [ ]:
# Activity 3 Try out

unique_countries = raw_df["Country_Region"].nunique()

print("Unique countries/regions:", unique_countries)

In [ ]:
# Activity 4 Try out

highest_missing = missing_summary.head(1)

print("Column with highest missingness:")
display(highest_missing)

print("Duplicate rows:", quality_checks["duplicate_rows"])

In [ ]:
# Activity 5 Try out

mean_confirmed = np.mean(confirmed_values)
median_confirmed = np.median(confirmed_values)
p90_confirmed = np.percentile(confirmed_values, 90)

print("Mean confirmed cases:", round(mean_confirmed, 2))
print("Median confirmed cases:", round(median_confirmed, 2))
print("90th percentile confirmed cases:", round(p90_confirmed, 2))

if mean_confirmed > median_confirmed:
    print("Signal: The distribution is right-skewed.")
else:
    print("Signal: The distribution is not strongly right-skewed.")

In [ ]:
# Activity 6 Try out

activity_top_n = 15
activity_country_to_view = "Brazil"
activity_min_confirmed_for_rate = 1_000_000

display_columns = [
    "country",
    "confirmed",
    "deaths",
    "case_fatality_percent"
]

activity_top_countries = (
    country_summary
    .sort_values("confirmed", ascending=False)
    .head(activity_top_n)
    .copy()
)

activity_selected_country = country_summary[
    country_summary["country"]
    .astype(str)
    .str.casefold()
    .eq(activity_country_to_view.casefold())
].copy()

activity_rate_table = (
    country_summary[
        country_summary["confirmed"] >= activity_min_confirmed_for_rate
    ]
    .sort_values("case_fatality_percent", ascending=False)
    .copy()
)

print("Activity TOP_N:", activity_top_n)
print("Activity country:", activity_country_to_view)
print("Activity minimum confirmed cases:", f"{activity_min_confirmed_for_rate:,}")

print("\nTop countries by confirmed cases:")
display(activity_top_countries[display_columns])

print(f"\nSelected country: {activity_country_to_view}")
if activity_selected_country.empty:
    print("Selected country was not found. Check the spelling or country name used in the dataset.")
else:
    display(activity_selected_country[display_columns])

print("\nCountries meeting the minimum confirmed-case threshold, sorted by case fatality percent:")
display(activity_rate_table[display_columns].head(10))

In [ ]:
# Activity 7 Try out

death_plot_data = (
    country_summary
    .sort_values("deaths", ascending=False)
    .head(TOP_N)
    .sort_values("deaths", ascending=True)
    .copy()
)

death_plot_data["deaths_thousands"] = death_plot_data["deaths"] / 1_000

plt.figure(figsize=(9, 5))

plt.barh(
    death_plot_data["country"],
    death_plot_data["deaths_thousands"],
    color=ALERT_RED
)

plt.title(f"Top {TOP_N} Countries by Reported COVID-19 Deaths")
plt.xlabel("Reported deaths, in thousands")
plt.ylabel("Country")
plt.tight_layout()
plt.show()

print("Highest reported deaths in this view:")
display(death_plot_data.tail(1)[["country", "deaths"]])

In [ ]:
# Activity 8 Try out

seaborn_activity_data = country_summary[
    (country_summary["confirmed"] >= 10_000) &
    (country_summary["case_fatality_percent"].notna())
].copy()

burden_order = ["Moderate", "High", "Very high"]
burden_order = [
    label for label in burden_order
    if label in seaborn_activity_data["case_burden_class"].unique()
]

plt.figure(figsize=(8, 5))

sns.boxplot(
    data=seaborn_activity_data,
    x="case_burden_class",
    y="case_fatality_percent",
    order=burden_order,
    color=CORAL
)

plt.title("Case Fatality Percent Across Burden Classes")
plt.xlabel("Case burden class")
plt.ylabel("Case fatality percent")
plt.tight_layout()
plt.show()

median_by_class = (
    seaborn_activity_data
    .groupby("case_burden_class")["case_fatality_percent"]
    .median()
    .reindex(burden_order)
    .round(2)
)

print("Median case fatality percent by burden class:")
print(median_by_class)

In [ ]:
# Activity 9 try out

# Step 1: Select numeric columns connected to the country-level dataset
activity_heatmap_columns = [
    "confirmed",
    "deaths",
    "recovered",
    "active",
    "case_fatality_percent",
    "recovery_percent",
    "active_percent"
]

# Step 2: Keep only columns that are actually present in the dataset
activity_heatmap_columns = [
    col for col in activity_heatmap_columns
    if col in country_summary.columns
]

# Step 3: Create a smaller dataset for the heatmap
activity_heatmap_data = country_summary[activity_heatmap_columns].copy()

# Step 4: Calculate the correlation matrix
activity_correlation_matrix = activity_heatmap_data.corr()

# Step 5: Plot the heatmap
plt.figure(figsize=(8, 6))

sns.heatmap(
    activity_correlation_matrix,
    annot=True,
    cmap="coolwarm",
    center=0,
    linewidths=0.5,
    fmt=".2f"
)

plt.title("Activity Heatmap: Relationships Between Country-Level COVID-19 Indicators")
plt.tight_layout()
plt.show()

# Step 6: Identify the strongest positive relationship, excluding self-correlation
upper_triangle = activity_correlation_matrix.where(
    np.triu(np.ones(activity_correlation_matrix.shape), k=1).astype(bool)
)

strongest_positive = (
    upper_triangle
    .stack()
    .sort_values(ascending=False)
    .head(1)
)

print("Strongest positive relationship in this activity heatmap:")
print(strongest_positive)

print("\nScientific interpretation:")
print("This heatmap shows descriptive relationships between variables.")
print("It does not prove that one variable causes another.")


In [ ]:
# Mini Case Study: Create a country-level review table

# Step 1: Keep only countries with enough confirmed cases for fairer rate comparison
review_base = country_summary[
    (country_summary["confirmed"] >= MIN_CONFIRMED_FOR_RATE)
    & (country_summary["case_fatality_percent"].notna())
].copy()

# Step 2: Use the median case fatality percent as a simple comparison cutoff
rate_cutoff = review_base["case_fatality_percent"].median()

# Step 3: Select countries with confirmed cases above the minimum threshold
# and case fatality percent at or above the comparison cutoff
review_table = (
    review_base[
        review_base["case_fatality_percent"] >= rate_cutoff
    ]
    .sort_values(
        ["confirmed", "case_fatality_percent"],
        ascending=[False, False]
    )
    .reset_index(drop=True)
)

# Step 4: Keep only the most relevant columns for interpretation
review_table = review_table[
    [
        "country",
        "reporting_units",
        "confirmed",
        "deaths",
        "case_fatality_percent",
        "active_share_percent",
        "case_burden_class"
    ]
]

print("Mini case study: Country-level review table")
print("Minimum confirmed cases used:", f"{MIN_CONFIRMED_FOR_RATE:,}")
print("Median case fatality percent cutoff:", round(rate_cutoff, 2))

display(review_table.head(15))


In [ ]:
# In the next cell you can try out

# Mini Case Study: Summarize one selected country

selected_country_summary = country_summary[
    country_summary["country"]
    .astype(str)
    .str.casefold()
    .eq(COUNTRY_TO_VIEW.casefold())
].copy()

if selected_country_summary.empty:
    print("Selected country not found. Change COUNTRY_TO_VIEW and rerun.")
else:
    row = selected_country_summary.iloc[0]

    print("Selected country summary")
    print("------------------------")
    print("Country:", row["country"])
    print("Reporting units:", int(row["reporting_units"]))
    print("Confirmed cases:", f"{int(row['confirmed']):,}")
    print("Reported deaths:", f"{int(row['deaths']):,}")
    print("Case fatality percent:", round(row["case_fatality_percent"], 2))
    print("Active share percent:", round(row["active_share_percent"], 2))
    print("Burden class:", row["case_burden_class"])

    print("\nInterpretation:")
    print("This is a descriptive summary based on reported data.")
    print("It should be used for review and discussion, not as direct proof of risk or causation.")
